# EEG datasets and recordings

REPLICA exposes ERP amplitudes, frequency-band summaries and continuous EEG
recordings where the source provides them. Their units and observation levels
differ: an ERP amplitude can be a signed voltage, whereas a continuous recording
contains a sequence of samples and channels. Inspect the selected loader and
measure before choosing a response transform or analysis.

The inventory below comes from the installed dataset catalog and updates when
loaders are added. `replica datasets --loadable` lists all keys,
and `replica datasets --describe NAME` explains one loader's options.

Szewczyk and Federmeier's pooled N400 data and Michaelov's N400 data provide
participant-level responses at a sentence's critical word. Their
`aggregate="mean"` option averages participants. Michaelov's
`aggregate="electrode"` view preserves individual electrode observations.
Frank et al.'s ERP loader exposes word-aligned amplitudes and optional continuous
recordings. Its `attach_predictability_norms()` method attaches the de Varda et al. cloze
and predictability norms; the UCL reading-time loaders share this method.

## Prerequisites

```bash
pip install -e ".[models,zuco]"
replica fetch zuco_n400 szewczyk_n400 federmeier2007_n400 michaelov_n400
```

The evaluation example requires the `models` extra. Native ZuCo MATLAB recordings
require `zuco`; continuous recordings can be large, so select participants when
loading them. `replica fetch frank2015_erp` retrieves the ERP archive, including
continuous EEG; the loader also uses UCL stimulus text from `replica fetch ucl_et`.

In [ ]:
import pandas as pd

from replica.dataset import available_datasets, describe_dataset, load_dataset

# Read the installed catalog without downloading or loading dataset files.
descriptions = [describe_dataset(name) for name in available_datasets()]
inventory = pd.DataFrame(
    [
        {"Dataset": d.name, "Loader": d.loader, "Measures": ", ".join(d.measures)}
        for d in descriptions
        if d.paradigm == "eeg"
    ]
)
inventory

In [ ]:
import numpy as np

from replica.dataset import describe_dataset

# Inspect valid aggregation levels and any pinned study selection.
print(describe_dataset("michaelov_n400"))

## ZuCo: choose an observation level

`zuco_n400` reads the **participant-averaged ERP table** distributed with
Kuribayashi et al. (2025). This table contains N400, P600 and ANT amplitudes.
It does not expose individual responses.

Individual ZuCo observations are available through `zuco1_eeg` and `zuco2_eeg`:
these loaders expose participant-level frequency-band power and, with
`continuous=True`, the released continuous EEG recordings. These are different
quantities from the averaged ERP table. See `describe_dataset("zuco2_eeg")`
for task and participant-selection options.

```python
individual = load_dataset("zuco2_eeg", task="NR", max_subjects=1, continuous=True)
recordings = individual.recordings(modality="eeg")
```

The following cells inspect the averaged table.

In [ ]:
zuco = load_dataset("zuco_n400")

print(f"Stimuli:      {len(zuco.stimuli())}")
print(f"Measures:     {sorted(zuco.measures)}")
print(f"Participants: {zuco.participants}")

stim = zuco.stimuli()[0]
n400 = next(
    d for d in zuco.data(measure="n400") if d.stimulus.stimulus_id == stim.stimulus_id
)
print(f"\nSentence: {' '.join(stim.unit_strings('words'))}")
print(f"Per-word N400 (uV): {np.round(n400.values, 2)}")

## Szewczyk and Federmeier: pooled or per source study

`szewczyk_n400` loads the five studies in the pooled release;
`SzewczykN400(study=...)` selects one. The catalog also registers each study,
such as `federmeier2007_n400`. The N400 observation is aligned to the critical
word in its sentence. `covariates()` exposes the available cloze probability,
word frequency, concreteness, orthographic-neighborhood and plausibility columns;
the set depends on the source study.

In [ ]:
mega = load_dataset("szewczyk_n400")  # study=None -> all five studies
print(
    f"Mega-N400:         {len(mega.stimuli())} items, {len(mega.participants)} subjects"
)

fed = load_dataset("federmeier2007_n400")  # study="Federmeier et al. (2007)"
print(
    f"Federmeier et al. (2007): {len(fed.stimuli())} items, {len(fed.participants)} subjects"
)

d = fed.data(measure="n400")[0]
cov = fed.covariates(d.stimulus)
tok = int(np.flatnonzero(~np.isnan(d.values))[0])  # the critical token
word = d.stimulus.unit_strings("words")[tok]
print(f"\nSentence: {' '.join(d.stimulus.unit_strings('words'))}")
print(
    f"N400 on '{word}': {d.values[tok]:.2f} uV  "
    f"(cloze={cov['cloze'][tok]:.3f}, log_freq={cov['log_frequency'][tok]:.2f})"
)

## Michaelov: cluster mean or per electrode

Use `aggregate="electrode"` to retain observations from each centroparietal
electrode and model electrode-specific random intercepts, as in
[Michaelov et al. (2024)](https://doi.org/10.1162/nol_a_00105).

In [ ]:
mich = load_dataset("michaelov_n400")
print(f"Items:      {len(mich.stimuli())}")
print(f"Subjects:   {len(mich.participants)}")
print(f"Conditions: {sorted({s.metadata.get('condition') for s in mich.stimuli()})}")

# Electrode identity is available on each Datum's metadata.
elec = load_dataset("michaelov_n400", aggregate="electrode")
electrodes = sorted({d.metadata.get("electrode") for d in elec.data(measure="n400")})
print(f"Electrodes: {electrodes}")

## Evaluating a predictor against the N400

This illustration tests GPT-2 surprisal beyond word length on the
participant-averaged `zuco_n400` table. It uses OLS and holds out sentences.
Participant random effects or reader holdout require individual observations;
the `zuco1_eeg` and `zuco2_eeg` loaders provide those at other EEG measurement
levels.

For an analysis with additional controls, inspect `zuco.default_controls()` and
`zuco.covariates(stimulus)`. This table includes the N400 baseline amplitude and
log word frequency. The example below is a minimal API illustration, not a
reproduction of a published statistical model.

In [ ]:
from replica import (
    OLS,
    AnalysisConfig,
    EvaluationSpec,
    ItemSplitter,
    ModelFeature,
    evaluate,
)
from replica.feature import HuggingFaceLM, Surprisal, UnitLength

spec = EvaluationSpec(
    predictors=[ModelFeature(HuggingFaceLM("gpt2"), Surprisal())],
    controls=[UnitLength()],
    targets=[zuco.feature("n400")],
)
results = evaluate(
    zuco, spec, AnalysisConfig(model=OLS(), design=None, cv=ItemSplitter(10))
)
print(f"Surprisal delta-log-likelihood on ZuCo N400: {results[0].delta_ll:.2f}")

## Response scale and alignment

Keep ERP amplitudes on their raw signed scale; a logarithmic transform cannot
represent negative amplitudes. Frequency-band power and continuous recordings
have their own measurement definitions.

The Szewczyk and Michaelov loaders align each critical-word amplitude to its
word and mark other word positions unavailable. Continuous EEG uses sample
times and event onsets. The [segmentation and reaggregation tutorial](../tutorials/segmentation.ipynb#eeg-and-fmri)
explains why text segmentation alone cannot create a new EEG response.